# Phase D v4 — bounded M6 Colab execution
Fresh study: run setup/checks, then separately authorize preflight and pilot.
**Lost runtime during the measured loop:** run the first four code cells (environment, persistence/helpers, source verification, tests). Then jump to **Restore the entire measured batch**, **Optionally reconstruct the ephemeral dataset cache**, and **Continue the restored batch**. Do not rerun warm-ups, donor or replay.
Recovery restores all completed conditions and the interrupted condition from the original Drive folder. Set the explicit compatible-T4 policy for a replacement T4. Data reconstruction and continuation retain separate default-disabled gates.
The setup cell mounts Drive and clones the originally pinned GitHub commit. The historical ZIP remains a separate Drive upload. Existing training sources and scientific fingerprints are preserved.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys

# Set before importing PyTorch or initializing CUDA.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Recreate the original checkout and mounted Drive after runtime deletion.
from google.colab import drive
drive.mount('/content/drive')
UPLOADS = Path('/content/drive/MyDrive/phase_d_v4_uploads')
COMMIT_FILE = UPLOADS/'colab_recovery_check_01_commit.txt'
ROOT = Path('/content/probabilistic backpropagation')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/Anupam-Dasgupta/probabilistic-backpropagation.git',
                    str(ROOT)], check=True)
    if COMMIT_FILE.is_file():
        subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', COMMIT_FILE.read_text().strip()], check=True)
commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if not COMMIT_FILE.exists():
    UPLOADS.mkdir(parents=True, exist_ok=True)
    COMMIT_FILE.write_text(commit + '\n')
else:
    assert COMMIT_FILE.read_text().strip() == commit, 'Checkout differs from original pinned commit'
import shutil
archive_source = UPLOADS/'cifar_d_resnet_v3.zip'
archive_local = ROOT/'cifar_d_resnet_v3.zip'
if not archive_local.exists():
    assert archive_source.is_file(), 'Original results archive is required in phase_d_v4_uploads'
    shutil.copyfile(archive_source, archive_local)
print('Original checkout:', commit)

import numpy as np, torch, torchvision
ROOT = Path('/content/probabilistic backpropagation')
assert ROOT.is_dir(), 'Upload or mount the repository at ROOT before continuing'
os.chdir(ROOT)
# Make test imports unambiguous even when an older clone lacks this package marker.
test_package = ROOT / 'tests/__init__.py'
if not test_package.exists():
    test_package.write_text('"""Project tests and shared synthetic helpers."""\n')
# Git normalizes this historical document to LF; restore only the exact audited bytes.
governing = json.loads((ROOT/'configs/phase_d_v4_reference_manifest.json').read_text())['governing_artifacts']
spec = ROOT/'RESEARCH_SPEC.md'
spec_bytes = spec.read_bytes()
if hashlib.sha256(spec_bytes).hexdigest() != governing['RESEARCH_SPEC.md']['sha256']:
    original_bytes = spec_bytes.replace(b'\r\n', b'\n').replace(b'\n', b'\r\n')
    assert hashlib.sha256(original_bytes).hexdigest() == governing['RESEARCH_SPEC.md']['sha256'], 'Specification content differs'
    spec.write_bytes(original_bytes)
def run_checked(arguments):
    """Run one repository command and stop this cell on any nonzero exit."""
    try:
        result = subprocess.run([sys.executable, '-B', *map(str, arguments)], check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.CalledProcessError as exc:
        print(exc.stdout or str(exc), flush=True)
        raise
    print(result.stdout or '', flush=True)
    return result
print(sys.version)
print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'numpy', np.__version__)
assert torch.cuda.is_available(), 'CUDA is required; CPU fallback is forbidden'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu, 'CUDA:', torch.version.cuda, 'cuDNN:', torch.backends.cudnn.version())
assert 'T4' in gpu, f'Expected an NVIDIA T4 before timing comparisons, got {gpu}'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)
assert not torch.backends.cuda.matmul.allow_tf32 and not torch.backends.cudnn.allow_tf32

In [ ]:
# Optional Drive persistence. Active training always stays under /content.
# Mount Drive explicitly before enabling persistence.
#
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import json
import shutil

PERSISTENCE_ROOT = Path('/content/drive/MyDrive/phase_d_v4_pilot_seed0')
# Example:
# PERSISTENCE_ROOT = Path(
#     '/content/drive/MyDrive/phase_d_v4_pilot_seed0'
# )


def persistence_arguments(run_id):
    """Return verified checkpoint/snapshot backup arguments for one run."""
    if PERSISTENCE_ROOT is None:
        return []
    return ['--persistence-dir', PERSISTENCE_ROOT / run_id]


def preserve_support(*paths):
    """Back up immutable study configs and split files with checksums."""
    if PERSISTENCE_ROOT is None:
        return

    from phase_d_v4_io import backup_study_support

    for path in paths:
        backup_study_support(path, STUDY, PERSISTENCE_ROOT)


def restore_support(path):
    """Restore one verified study support file without overwriting changes."""
    if PERSISTENCE_ROOT is None:
        raise RuntimeError("Persistence is not enabled")

    from phase_d_v4_io import restore_study_support
    return restore_study_support(path, STUDY, PERSISTENCE_ROOT)


# Batch orchestration lives in the notebook so existing checkpoint source
# fingerprints remain valid. Training, optimizer and checkpoint code is unchanged.
from phase_d_v4_io import (
    STRICT_RESUME_POLICY, COMPATIBLE_T4_RESUME_POLICY,
    atomic_write_json, load_run_config, load_checkpoint,
    latest_committed_checkpoint, sha256_file,
)


def batch_entries():
    """Validate the saved order and return its measured run definitions.

    Reads global pilot metadata only; returns an ordered list of mappings.
    Duplicate, missing or unsafe IDs raise before any files or training change.
    """
    order = pilot['run_order']['measured_block']
    entries = pilot['measured_conditions']
    ids = [item['run_id'] for item in entries]
    if len(set(ids)) != len(ids) or len(set(order)) != len(order) or set(ids) != set(order):
        raise ValueError('Saved measured order is inconsistent')
    for run_id in ids + [item['run_id'] for item in pilot['dependency_phase']]:
        if not run_id or Path(run_id).name != run_id or '/' in run_id or '\\' in run_id or run_id in {'.', '..'}:
            raise ValueError('Unsafe saved run ID')
    lookup = {item['run_id']: item for item in entries}
    return [lookup[run_id] for run_id in order]


def batch_run_status(run_id):
    """Read checkpoint-backed completion for one run; never trust a summary alone.

    Input is a saved run ID. Returns complete, partial or not_started. Existing
    runs must match their config, summary and checksummed checkpoint. A lost
    local run with Drive evidence, or a recorded command failure, raises rather
    than silently starting over. Reads files/CPU tensors only; writes nothing.
    """
    run = STUDY / run_id
    remote = PERSISTENCE_ROOT / run_id
    if (PERSISTENCE_ROOT / '_batch_status' / f'{run_id}.failed.json').exists():
        raise RuntimeError(f'Recorded failure needs investigation before retry: {run_id}')
    if not run.exists():
        if remote.exists() and any(remote.iterdir()):
            raise RuntimeError(f'Drive state exists for {run_id}; run whole-batch restoration first')
        if (PERSISTENCE_ROOT / '_batch_status' / f'{run_id}.started.json').exists():
            raise RuntimeError(f'{run_id} started but has no recoverable checkpoint; stop for review')
        return 'not_started'
    config = load_run_config(CONFIGS / f'{run_id}.json')
    payload = load_checkpoint(
        latest_committed_checkpoint(run / 'checkpoints'),
        expected={'run_id': run_id, 'resolved_config': config.to_dict()},
    )
    summary = json.loads((run / 'summary.json').read_text())
    state = payload['run_state']
    if (summary['completed'] != state['completed'] or
            summary['checkpoint_sequence'] != payload['checkpoint_sequence']):
        raise ValueError(f'Summary disagrees with committed checkpoint: {run_id}')
    return 'complete' if state['completed'] else 'partial'


def verify_batch_replay():
    """Check completed donor/replay equality using the actual flat counter schema.

    Reads the two dependency runs and checksum-verified CPU checkpoints. Returns
    a report only when models, optimizers, masks, counters and data order match
    exactly. Missing fields, invalid runs or differences raise; no state changes.
    """
    from analyze_phase_d_v4 import validate_run_directory, _state_tree_equal
    donor = validate_run_directory(DONOR_RUN)
    replay_run = STUDY / 'seed0_relative_replay_learned_equivalence'
    replay = validate_run_directory(replay_run)
    if donor['method'] != 'relative_feedback' or replay['method'] != 'relative_replay_learned':
        raise ValueError('Wrong replay dependency methods')
    if donor['seed'] != replay['seed'] or donor['hardware_key'] != replay['hardware_key']:
        raise ValueError('Replay dependencies differ in seed or original hardware')
    left = load_checkpoint(latest_committed_checkpoint(DONOR_RUN / 'checkpoints'))
    right = load_checkpoint(latest_committed_checkpoint(replay_run / 'checkpoints'))
    a, b = left['run_state'], right['run_state']
    counters = (
        'global_step', 'postwarm_step', 'exact_trunk_backwards',
        'synthetic_backwards', 'zero_backwards', 'skipped_slots',
        'trunk_optimizer_steps', 'head_optimizer_steps',
        'predictor_fits', 'predictor_inferences',
    )
    checks = {key: _state_tree_equal(left[key], right[key])
              for key in ('model', 'predictor', 'optimizers')}
    checks.update({key: a[key] == b[key] for key in (
        'executed_exact_mask', 'executed_head_mask', 'data_order_hashes',
    )})
    checks['counters'] = all(a[key] == b[key] for key in counters)
    if not all(checks.values()):
        raise ValueError(f'Replay mismatch: {[key for key, equal in checks.items() if not equal]}')
    return {'equivalent': True, 'checks': checks}


def verify_batch_progress():
    """Check that completed/partial runs form a valid prefix of the frozen order.

    Returns ordered (entry, status) pairs after checking both finished dependency
    runs and the retained pre-block profile. An out-of-order run or more than
    one interrupted run raises. No data loading, training or writes occur.
    """
    if PERSISTENCE_ROOT is None or not PERSISTENCE_ROOT.is_dir():
        raise RuntimeError('Original Drive persistence root is required')
    entries = batch_entries()
    for item in pilot['dependency_phase']:
        if batch_run_status(item['run_id']) != 'complete':
            raise RuntimeError('Measured-batch recovery requires completed donor and replay dependencies')
    for name in ('profile_summary.json', 'profile_samples.csv'):
        if not (STUDY / 'pre_block_profile' / name).is_file():
            raise FileNotFoundError('Original pre-block profile is required; do not regenerate it')
    progress = [(entry, batch_run_status(entry['run_id'])) for entry in entries]
    unfinished_seen = False
    for entry, status in progress:
        if unfinished_seen and status != 'not_started':
            raise ValueError(f'Run state violates saved order: {entry["run_id"]}')
        if status != 'complete':
            unfinished_seen = True
    return progress


def restore_measured_batch(policy=STRICT_RESUME_POLICY, device='cuda'):
    """Restore every saved dependency and measured run, without training.

    Reads global study/Drive paths and the saved order. Restores checksummed
    support files and full retained checkpoint sets, preserving completed runs.
    Compatibility is checked before copying each run; source changes, corrupt
    snapshots and missing started-run state fail closed. Returns a status list.
    Uses the existing strict/T4 policies; records recovery duration separately.
    """
    import time, uuid
    from gradient_phase_d_resnet_v4 import configure_determinism, _resolved_artifacts
    from phase_d_v4_io import (
        restore_study_support, verify_resume_snapshot_execution,
        restore_or_verify_resume_snapshot, environment_metadata,
    )
    global pilot
    if PERSISTENCE_ROOT is None or not PERSISTENCE_ROOT.is_dir():
        raise RuntimeError('Mount original Drive and set PERSISTENCE_ROOT first')
    started = time.perf_counter()
    for relative in ('run_order_manifest.json', 'split_indices.npz',
                     'split_indices.npz.manifest.json'):
        restore_study_support(STUDY / relative, STUDY, PERSISTENCE_ROOT)
    pilot = json.loads((STUDY / 'run_order_manifest.json').read_text())
    entries = [*pilot['dependency_phase'], *batch_entries()]
    for item in entries:
        run_id = item['run_id']
        restore_study_support(CONFIGS / f'{run_id}.json', STUDY, PERSISTENCE_ROOT)
    for name in ('profile_summary.json', 'profile_samples.csv'):
        restore_study_support(STUDY / 'pre_block_profile' / name, STUDY, PERSISTENCE_ROOT)
    reports = []
    for item in entries:
        run_id = item['run_id']
        remote = PERSISTENCE_ROOT / run_id
        if not remote.exists() or not any(remote.iterdir()):
            continue  # Status validation below distinguishes unstarted from lost work.
        config = load_run_config(CONFIGS / f'{run_id}.json')
        configure_determinism(config, device)
        verified = verify_resume_snapshot_execution(
            remote, current_config=config,
            current_environment=environment_metadata(device), policy=policy,
        )
        if verified['artifacts'] != _resolved_artifacts(verified['artifacts']):
            raise ValueError('Training source files differ from the saved run; restore the original code')
        # This copies complete runs too: the loop must not mistake them for new work.
        # Windows may briefly hold a directory handle during the atomic rename.
        # Retry only a failed restoration into an absent destination, never a
        # training command, checksum failure, or existing local run. Every
        # attempt repeats the original full snapshot verification.
        import os
        for attempt in range(3):
            try:
                restore_or_verify_resume_snapshot(remote, STUDY / run_id)
                break
            except PermissionError:
                if os.name != 'nt' or (STUDY / run_id).exists() or attempt == 2:
                    raise
                time.sleep(0.2 * (attempt + 1))
        reports.append({'run_id': run_id, 'compatibility': verified['resume_compatibility']})
        print('Restored/verified:', run_id, flush=True)
    progress = verify_batch_progress()
    print(json.dumps(verify_batch_replay(), indent=2))
    record = {
        'scope': 'notebook_batch_restoration',
        'seconds': time.perf_counter() - started,
        'policy': policy, 'runs': reports,
        'unknown_interruption_gap_seconds': None,
        'clean_single_session_timing': False,
    }
    record_path = STUDY / 'batch_recovery' / f'{uuid.uuid4().hex}.json'
    atomic_write_json(record_path, record)
    preserve_support(record_path)
    for entry, status in progress:
        print(status, entry['run_id'], flush=True)
    return progress


def run_measured_batch(*, authorized=False, policy=STRICT_RESUME_POLICY, device='cuda'):
    """Run remaining conditions once, in order, using existing resume mechanics.

    Explicit authorization is required. Completed checkpoints are verified and
    skipped; a partial run resumes with the selected strict/T4 policy. All
    commands retain Drive persistence and donor masks. Command/check failures
    are saved to Drive and stop the batch; no automatic retries occur. Returns
    nothing. Starts training subprocesses and writes run/status files.
    """
    if not authorized:
        raise PermissionError('Batch continuation is default-disabled')
    progress = verify_batch_progress()
    print(json.dumps(verify_batch_replay(), indent=2))
    from phase_d_v4_io import REPLAY_METHODS, SOURCE_DERIVED_SCHEDULE_METHODS
    source_methods = {method.value for method in REPLAY_METHODS | SOURCE_DERIVED_SCHEDULE_METHODS}
    for position, (entry, status) in enumerate(progress, 1):
        run_id = entry['run_id']
        run = STUDY / run_id
        print(f'[{position}/{len(progress)}] {status}: {run_id}', flush=True)
        try:
            if status != 'complete':
                command = 'resume' if status == 'partial' else 'run'
                arguments = [
                    'gradient_phase_d_resnet_v4.py', command,
                    '--authorize-real-data', '--authorize-full-pilot',
                    '--device', device, '--config', CONFIGS / f'{run_id}.json',
                    '--data-dir', DATA, '--split-file', SPLIT, '--out', run,
                    *persistence_arguments(run_id),
                ]
                if command == 'resume':
                    arguments += ['--checkpoint', latest_committed_checkpoint(run / 'checkpoints'),
                                  '--resume-compatibility', policy]
                if entry['method'] in source_methods:
                    arguments += ['--mask', DONOR_MASK, '--mask-manifest', DONOR_MANIFEST]
                marker = PERSISTENCE_ROOT / '_batch_status' / f'{run_id}.started.json'
                if not marker.exists():
                    atomic_write_json(marker, {'run_id': run_id, 'config_sha256': sha256_file(CONFIGS / f'{run_id}.json')})
                run_checked(arguments)
                if batch_run_status(run_id) != 'complete':
                    raise RuntimeError(f'Command returned without completing {run_id}')
            if entry['method'] == 'relative_feedback':
                run_checked(['analyze_phase_d_v4.py', 'compare-feedback-masks', DONOR_RUN, run])
        except Exception as exc:
            atomic_write_json(
                PERSISTENCE_ROOT / '_batch_status' / f'{run_id}.failed.json',
                {'run_id': run_id, 'error': str(exc), 'automatic_retry': False},
            )
            raise
    print('All measured conditions complete. Run final validation/drift cell next.', flush=True)

In [ ]:
# A Git clone omits the historical results. Upload the existing archive to ROOT; never download or regenerate it here.
import zipfile
reference_manifest = json.loads(Path('configs/phase_d_v4_reference_manifest.json').read_text())
archive_identity = reference_manifest['reference_results_archive']
archive_path = Path(archive_identity['path'])
results_path = Path(reference_manifest['reference_results']['path'])
# Migrate only the independently verified repackaging of the original 44 files.
# Their frozen results-tree identity is still required by the test below.
from phase_d_v4_io import sha256_file
verified_packaging = 'fd52ed7f6332d7ad7f5ad7ff913980f64169d6cba2aed9d8cfe3d1f46c81ea22'
if archive_path.is_file() and sha256_file(archive_path) == verified_packaging:
    archive_identity['sha256'] = verified_packaging
    Path('configs/phase_d_v4_reference_manifest.json').write_text(json.dumps(reference_manifest, indent=2) + '\n')
if not results_path.is_dir():
    assert archive_path.is_file(), f'Upload {archive_path.name} into repository root before tests'
    from phase_d_v4_io import sha256_file
    assert sha256_file(archive_path) == archive_identity['sha256']
    with zipfile.ZipFile(archive_path) as bundle:
        names = bundle.namelist()
        assert names and all(name.startswith(archive_identity['required_top_level']) for name in names)
        assert all('..' not in Path(name).parts and not Path(name).is_absolute() for name in names)
        bundle.extractall(ROOT)
run_checked(['-m','unittest','discover','-s','tests','-p','test_phase_d_v4.py','-k','test_reference_results_tree_matches_manifest','-v'])
for name, record in reference_manifest['governing_artifacts'].items():
    digest = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert digest == record['sha256'], (name, digest, record['sha256'])
for name in ['gradient_phase_d_resnet_v4.py', 'phase_d_v4_io.py', 'phase_d_v4_diagnostics.py', 'phase_d_v4_analysis.py', 'analyze_phase_d_v4.py', 'configs/phase_d_v4_reference.json', 'configs/phase_d_v4_pilot_seed0_manifest.json']:
    print(name, hashlib.sha256(Path(name).read_bytes()).hexdigest())
print('Protected identities verified')


In [ ]:
run_checked(['run_phase_d_v4_tests.py'])
run_checked(['gradient_phase_d_resnet_v4.py', 'audit', '--device', 'cuda'])


In [ ]:
SMOKE_RESUMED = Path('/content/phase_d_v4_smoke_resumed')
SMOKE_FULL = Path('/content/phase_d_v4_smoke_uninterrupted')
assert not SMOKE_RESUMED.exists() and not SMOKE_FULL.exists(), 'Use fresh smoke directories'
run_checked(['gradient_phase_d_resnet_v4.py', 'smoke', '--device', 'cuda', '--out', SMOKE_RESUMED, '--stop-after-global-epoch', '0'])
partial = json.loads((SMOKE_RESUMED/'summary.json').read_text())
assert partial['completed'] is False and partial['checkpoint_sequence'] == 1
run_checked(['gradient_phase_d_resnet_v4.py', 'resume', '--device', 'cuda', '--out', SMOKE_RESUMED])
run_checked(['gradient_phase_d_resnet_v4.py', 'smoke', '--device', 'cuda', '--out', SMOKE_FULL])
run_checked(['analyze_phase_d_v4.py', 'compare-resume', SMOKE_FULL, SMOKE_RESUMED])
resumed_manifest = json.loads((SMOKE_RESUMED/'manifest.json').read_text())
resumed_summary = json.loads((SMOKE_RESUMED/'summary.json').read_text())
assert resumed_manifest['resumed'] is True and resumed_summary['completed'] is True
assert resumed_summary['checkpoint_sequence'] == 2
print('Synthetic resumed state equals uninterrupted state; timing and accuracy are not research evidence')


In [ ]:
AUTHORIZE_REAL_DATA_PREFLIGHT = False
if not AUTHORIZE_REAL_DATA_PREFLIGHT:
    raise RuntimeError('STOP: real-data preflight is disabled by default; obtain approval, set this value True, and rerun this cell')
# This approval covers dataset setup, two four-epoch warm-ups (eight full training-split passes), and bounded profiling only.
STUDY = Path('/content/runs_phase_d_v4/phase_d_v4_pilot_seed0')
CONFIGS = STUDY/'configs'
DATA = Path('/content/data')
SPLIT = STUDY/'split_indices.npz'
assert not STUDY.exists(), 'Use an explicit fresh study directory'
CONFIGS.mkdir(parents=True)
run_checked(['gradient_phase_d_resnet_v4.py', 'prepare-data', '--authorize-real-data', '--config', 'configs/phase_d_v4_reference.json', '--data-dir', DATA, '--split-file', SPLIT, '--download'])
RUN_ORDER_MANIFEST = STUDY/'run_order_manifest.json'
from phase_d_v4_io import copy_file_verified
copy_file_verified(Path('configs/phase_d_v4_pilot_seed0_manifest.json'), RUN_ORDER_MANIFEST)
preserve_support(
    SPLIT,
    SPLIT.with_suffix(SPLIT.suffix + '.manifest.json'),
    RUN_ORDER_MANIFEST,
)


In [ ]:
DONOR_ID = 'seed0_relative_feedback_donor'
DONOR_CONFIG = CONFIGS/f'{DONOR_ID}.json'
DONOR_RUN = STUDY/DONOR_ID
run_checked(['gradient_phase_d_resnet_v4.py', 'make-config', '--method', 'relative_feedback', '--run-id', DONOR_ID, '--output-root', STUDY, '--out-config', DONOR_CONFIG])
preserve_support(DONOR_CONFIG)
# Stop after global epoch 3: four warm-up epochs only, before calibration/post-warm training.
run_checked(['gradient_phase_d_resnet_v4.py', 'run', '--authorize-real-data', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', DONOR_RUN, '--stop-after-global-epoch', '3', *persistence_arguments(DONOR_ID)])
WARMUP_CKPT = DONOR_RUN/'checkpoints/checkpoint_000004.pt'
assert WARMUP_CKPT.is_file()
PLAIN_ID = 'seed0_exact_plain_preflight'
PLAIN_CONFIG = CONFIGS/f'{PLAIN_ID}.json'
PLAIN_RUN = STUDY/PLAIN_ID
run_checked(['gradient_phase_d_resnet_v4.py', 'make-config', '--method', 'exact', '--run-id', PLAIN_ID, '--output-root', STUDY, '--out-config', PLAIN_CONFIG])
preserve_support(PLAIN_CONFIG)
run_checked(['gradient_phase_d_resnet_v4.py', 'run', '--authorize-real-data', '--device', 'cuda', '--config', PLAIN_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', PLAIN_RUN, '--stop-after-global-epoch', '3', *persistence_arguments(PLAIN_ID)])
assert (PLAIN_RUN/'checkpoints/checkpoint_000004.pt').is_file()
print('Preflight warm-ups completed: 2 x 4 epochs = 8 full training-split passes')


In [ ]:
PREFLIGHT = STUDY/'resource_preflight'
run_checked(['gradient_phase_d_resnet_v4.py', 'preflight', '--authorize-real-data', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--checkpoint', WARMUP_CKPT, '--out', PREFLIGHT])
preflight = json.loads((PREFLIGHT/'profile_summary.json').read_text())
assert 'T4' in preflight['environment']['device_name']
required_components = {'plain_exact','exact_plus_predictor','learned_synthetic','zero_synthetic_shadow','skip_synthetic_shadow','controller_overhead'}
assert required_components.issubset(preflight['components'])
preserve_support(PREFLIGHT/'profile_samples.csv', PREFLIGHT/'profile_summary.json')
print(json.dumps(preflight['components'], indent=2))


In [ ]:
pilot = json.loads(Path('configs/phase_d_v4_pilot_seed0_manifest.json').read_text())
methods = [item['method'] for item in pilot['measured_conditions']]
expected_methods = ['exact','exact_standard_bn','relative_feedback','scheduled','relative_replay_zero_shadow','relative_replay_skip_shadow','relative_replay_exact_trunk','relative_periodic_epoch','relative_gap_permutation_epoch','relative_periodic_total','fixed_frontloaded']
assert methods == expected_methods
assert pilot['trajectory_accounting']['actual_full_trajectory_count'] == 13
assert len(pilot['run_order']['measured_block']) == 11
assert set(pilot['run_order']['measured_block']) == {item['run_id'] for item in pilot['measured_conditions']}
assert pilot['dependency_phase'][1]['depends_on'] == [DONOR_ID]
components = preflight['components']
slot_medians = [components[name]['median'] for name in ['plain_exact','exact_plus_predictor','learned_synthetic','zero_synthetic_shadow','skip_synthetic_shadow']]
slots = pilot['scientific_scope']['postwarm_epochs'] * pilot['scientific_scope']['training_batches_per_epoch']
trajectories = pilot['trajectory_accounting']['actual_full_trajectory_count']
predictor_warmup = json.loads((DONOR_RUN/'timing.json').read_text())['totals']['workflow_seconds']
plain_warmup = json.loads((PLAIN_RUN/'timing.json').read_text())['totals']['workflow_seconds']
actual_independent_warmups = pilot['trajectory_accounting']['plain_warmup_full_trajectory_attribution_count']*plain_warmup + pilot['trajectory_accounting']['predictor_warmup_full_trajectory_attribution_count']*predictor_warmup
scenario_low = actual_independent_warmups + trajectories * slots * min(slot_medians) / 1000
scenario_high = actual_independent_warmups + trajectories * slots * max(slot_medians) / 1000
checkpoint_bytes = preflight['checkpoint_size_bytes']
checkpoint_count = trajectories*pilot['scientific_scope']['total_epochs'] + pilot['scientific_scope']['warmup_epochs']
checkpoint_core_bytes = checkpoint_count*checkpoint_bytes
estimate = {
 'status':'preflight_scenarios_not_runtime_bounds',
 'actual_study_gpu_hours_scenario_low':scenario_low/3600,
 'actual_study_gpu_hours_scenario_high':scenario_high/3600,
 'standalone_method_costs':'reported separately from actual study resource use',
 'preflight_work_already_performed':'two four-epoch warm-ups = eight full training-split passes, plus bounded profile',
 'checkpoint_core_storage_gib_approx':checkpoint_core_bytes/2**30,
 'storage_assumption':f'current runner keeps all {checkpoint_count} epoch checkpoints; measured warm-up checkpoint size is a proxy',
 'storage_not_included':['resolved configs','split artifacts','masks','CSV/JSON logs','final predictors','analysis plots','checkpoint growth','Drive filesystem overhead'],
 'runtime_not_included_or_uncertain':['dataset setup','validation','calibration','checkpoint/log/Drive IO','finalization','hardware drift','thermal effects','mask-dependent slot mix'],
 'not_guaranteed_bounds':True, 'counts_are_not_flops':True}
print(json.dumps(pilot, indent=2))
print(json.dumps(estimate, indent=2))


## Full-pilot approval gate — stop here
Share the exact manifest, T4 record, verified resume smoke, component medians, scenario estimates, storage assumptions, and omissions. These are estimates, not guaranteed runtime bounds. Real-data preflight approval does not authorize donor completion or any full trajectory. Five seeds, optional ablations, retries, and sweeps remain unauthorized.

In [ ]:
AUTHORIZE_FULL_PILOT = False
if not AUTHORIZE_FULL_PILOT:
    raise RuntimeError('STOP: full pilot is disabled by default; obtain approval, set this value True, and rerun this cell')
run_checked(['gradient_phase_d_resnet_v4.py', 'resume', '--authorize-real-data', '--authorize-full-pilot', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', DONOR_RUN, *persistence_arguments(DONOR_ID)])
DONOR_MASK = DONOR_RUN/'masks.npz'
DONOR_MANIFEST = DONOR_RUN/'mask_manifest.json'
assert json.loads((DONOR_RUN/'summary.json').read_text())['completed'] is True
preserve_support(DONOR_MASK, DONOR_MANIFEST)


In [ ]:
# Resolve configs, run learned replay, verify equivalence, and profile before the measured block.
conditions = [
 ('exact','seed0_exact_frozen',None,None), ('exact_standard_bn','seed0_exact_standard_bn',None,None),
 ('relative_feedback','seed0_relative_feedback_measured',None,None), ('scheduled','seed0_scheduled',None,None),
 ('relative_replay_learned','seed0_relative_replay_learned_equivalence',None,None),
 ('relative_replay_zero_shadow','seed0_relative_replay_zero_shadow',None,None),
 ('relative_replay_skip_shadow','seed0_relative_replay_skip_shadow',None,None),
 ('relative_replay_exact_trunk','seed0_relative_replay_exact_trunk',None,None),
 ('relative_periodic_epoch','seed0_relative_periodic_epoch',None,None),
 ('relative_gap_permutation_epoch','seed0_relative_gap_permutation_epoch_r0',None,0),
 ('relative_periodic_total','seed0_relative_periodic_total',None,None),
 ('fixed_frontloaded','seed0_fixed_frontloaded_learned','learned',None),
]
source_methods = {name for name, *_ in conditions if name.startswith('relative_replay_') or name in {'relative_periodic_epoch','relative_gap_permutation_epoch','relative_periodic_total'}}
from gradient_phase_d_resnet_v4 import build_condition_config
for method, run_id, behavior, replica in conditions:
    if method == 'relative_replay_zero_shadow': behavior = 'zero'
    if method == 'relative_replay_skip_shadow': behavior = 'skip'
    config = build_condition_config(
        method, run_id=run_id, output_root=str(STUDY),
        synthetic_behavior=behavior, mask_replicate=replica,
        donor_mask_path=DONOR_MASK if method in source_methods else None,
        donor_manifest_path=DONOR_MANIFEST if method in source_methods else None,
    )
    destination = CONFIGS / f'{run_id}.json'
    if destination.exists():
        assert json.loads(destination.read_text()) == config.to_dict(), 'Existing config differs'
    else:
        atomic_write_json(destination, config.to_dict())
    preserve_support(destination)
REPLAY_ID = 'seed0_relative_replay_learned_equivalence'
REPLAY_RUN = STUDY/REPLAY_ID
run_checked(['gradient_phase_d_resnet_v4.py','run','--authorize-real-data','--authorize-full-pilot','--device','cuda','--config',CONFIGS/f'{REPLAY_ID}.json','--data-dir',DATA,'--split-file',SPLIT,'--out',REPLAY_RUN,'--mask',DONOR_MASK,'--mask-manifest',DONOR_MANIFEST,*persistence_arguments(REPLAY_ID)])
print(json.dumps(verify_batch_replay(), indent=2))
PRE_BLOCK = STUDY/'pre_block_profile'
run_checked(['gradient_phase_d_resnet_v4.py','preflight','--authorize-real-data','--device','cuda','--config',DONOR_CONFIG,'--data-dir',DATA,'--split-file',SPLIT,'--checkpoint',WARMUP_CKPT,'--out',PRE_BLOCK])
preserve_support(PRE_BLOCK/'profile_samples.csv', PRE_BLOCK/'profile_summary.json')

In [ ]:
"""Run all remaining measured conditions in the frozen order."""
# Finish the donor/replay/pre-block cell first. This cell never restores a lost study implicitly.
run_measured_batch(authorized=AUTHORIZE_FULL_PILOT)

In [ ]:
"""Restore the entire measured batch after a Colab runtime reset."""
# Run environment/source checks and the persistence helper cell first.
# No manual run ID is needed. Complete runs and the interrupted run are restored.
STUDY = Path('/content/runs_phase_d_v4/phase_d_v4_pilot_seed0')
CONFIGS = STUDY / 'configs'
DATA = Path('/content/data')
SPLIT = STUDY / 'split_indices.npz'
RUN_ORDER_MANIFEST = STUDY / 'run_order_manifest.json'
DONOR_ID = 'seed0_relative_feedback_donor'
DONOR_RUN = STUDY / DONOR_ID
DONOR_CONFIG = CONFIGS / f'{DONOR_ID}.json'
DONOR_MASK = DONOR_RUN / 'masks.npz'
DONOR_MANIFEST = DONOR_RUN / 'mask_manifest.json'
WARMUP_CKPT = DONOR_RUN / 'checkpoints/checkpoint_000004.pt'
PRE_BLOCK = STUDY / 'pre_block_profile'
RECOVERY_COMPATIBILITY = STRICT_RESUME_POLICY  # Choose COMPATIBLE_T4_RESUME_POLICY for a replacement T4.
recovered_progress = restore_measured_batch(policy=RECOVERY_COMPATIBILITY)
# The original donor config defines the common CIFAR split; no run choice is needed.
RUN_CONFIG = DONOR_CONFIG
RUN_DIR = DONOR_RUN
print('Whole batch restored. Run dataset recovery, then batch continuation.')

In [ ]:
"""Optionally reconstruct the ephemeral dataset cache after restoration."""
from torchvision import datasets
AUTHORIZE_RECOVERY_DATA_SETUP = False
assert RUN_DIR.is_dir(), 'Run the verified restoration cell first'
try:
    datasets.CIFAR10(root=str(DATA), train=True, download=False)
    print('Existing CIFAR-10 training cache verified')
except RuntimeError:
    if not AUTHORIZE_RECOVERY_DATA_SETUP:
        raise RuntimeError('STOP: dataset reconstruction remains default-disabled')
    TEMP_SPLIT = STUDY/'_recovery_split_check.npz'
    if TEMP_SPLIT.exists(): TEMP_SPLIT.unlink()
    temp_manifest = TEMP_SPLIT.with_suffix(TEMP_SPLIT.suffix + '.manifest.json')
    if temp_manifest.exists(): temp_manifest.unlink()
    run_checked(['gradient_phase_d_resnet_v4.py','prepare-data','--authorize-real-data','--config',RUN_CONFIG,'--data-dir',DATA,'--split-file',TEMP_SPLIT,'--download'])
    original = json.loads(SPLIT.with_suffix(SPLIT.suffix + '.manifest.json').read_text())
    regenerated = json.loads(temp_manifest.read_text())
    for key in ('training_indices_sha256','validation_indices_sha256'):
        assert original[key] == regenerated[key], 'Regenerated dataset split does not match original'
    print('Dataset cache reconstructed and split identities verified')


In [ ]:
"""Continue the restored batch through all remaining conditions."""
AUTHORIZE_RECOVERY_CONTINUATION = False
if not AUTHORIZE_RECOVERY_CONTINUATION:
    raise PermissionError('STOP: recovered scientific continuation remains default-disabled')
assert RUN_CONFIG.is_file() and SPLIT.is_file()
from torchvision import datasets
datasets.CIFAR10(root=str(DATA), train=True, download=False)
run_measured_batch(authorized=True, policy=RECOVERY_COMPATIBILITY)

In [ ]:
# Read-only validation and the required post-block drift profile after all saved-order runs finish.
completed = [str(path.parent) for path in STUDY.glob('*/summary.json') if json.loads(path.read_text()).get('completed')]
MEASURED = [path for path in completed if Path(path).name in set(pilot['run_order']['measured_block'])]
assert len(MEASURED) == len(pilot['run_order']['measured_block'])
run_checked(['analyze_phase_d_v4.py','validate',*MEASURED])
# A replacement runtime cannot supply an original-session drift measurement.
# Preserve the pre-block profile; do not replace it or label this a clean timing block.
from phase_d_v4_io import environment_metadata, execution_fingerprint, load_run_config
from gradient_phase_d_resnet_v4 import configure_determinism
donor_config = load_run_config(DONOR_CONFIG)
configure_determinism(donor_config, 'cuda')
profile_environment = json.loads((PRE_BLOCK/'profile_summary.json').read_text())['environment']
same_profile_environment = (
    execution_fingerprint(donor_config, profile_environment)
    == execution_fingerprint(donor_config, environment_metadata('cuda'))
)
recovered_block = (STUDY/'batch_recovery').exists()
if not same_profile_environment or recovered_block:
    print(json.dumps({
        'drift_status': 'unavailable_for_original_session_after_recovery',
        'clean_single_session_timing': False,
        'action': 'Retain results for accuracy analysis; report recovery and timing limitations. No automatic rerun.',
    }, indent=2))
else:
    POST_BLOCK = STUDY/'post_block_profile'
    run_checked(['gradient_phase_d_resnet_v4.py','preflight','--authorize-real-data','--device','cuda','--config',DONOR_CONFIG,'--data-dir',DATA,'--split-file',SPLIT,'--checkpoint',WARMUP_CKPT,'--out',POST_BLOCK])
    import csv
    from analyze_phase_d_v4 import hardware_drift_check
    with (PRE_BLOCK/'profile_samples.csv').open() as handle: pre_samples = [float(row['latency_ms']) for row in csv.DictReader(handle) if row['component'] == 'plain_exact']
    with (POST_BLOCK/'profile_samples.csv').open() as handle: post_samples = [float(row['latency_ms']) for row in csv.DictReader(handle) if row['component'] == 'plain_exact']
    drift = hardware_drift_check(pre_samples, post_samples)
    print(json.dumps(drift, indent=2))
    assert not drift['flagged'], 'Hardware drift exceeded 10%; retain all records and seek authorization before repeating the entire block'
# Use a fresh analysis directory; one seed produces insufficient-sample intervals by design.
# run_checked(['analyze_phase_d_v4.py','summarize','--out',STUDY/'analysis',*MEASURED])